Compute the six month downside deviation of stocks for each quarter

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_parquet('../data/ohlcv2.parquet')

In [3]:
data = data[['date', 'co_name', 'close']]

In [4]:
data = data.sort_values(['date', 'co_name'])
data['daily_return'] = data.groupby('co_name')['close'].pct_change()

In [5]:
# Define rolling window (6 months ≈ 126 trading days)
window = 126
target = 0

def downside_deviation(returns):
    """
    Only considers returns below the target (default 0).
    """
    downside_returns = np.where(returns < target, returns - target, 0)
    return np.sqrt(np.mean(downside_returns ** 2))

data['downside_deviation'] = data.groupby('co_name')['daily_return'].rolling(
    window=window, min_periods=window
).apply(downside_deviation, raw=True).reset_index(level=0, drop=True)


In [6]:
# How will downside deviation be used?
# On Feb 15th, probabilities of stocks will be available. This info will be combined with downside deviation for pf creation.
# We would like the 6 month downside deviation of stocks computed on Feb 14th.

In [7]:
# Define quarter cutoff dates
quarter_cutoffs = [
    (2, 14),   # February 14
    (5, 30),   # May 30
    (8, 14),   # August 14
    (11, 14)   # November 14
]

# Convert date to datetime if not already
data['date'] = pd.to_datetime(data['date'])

# Get unique years from the data
years = data['date'].dt.year.unique()

# Create all cutoff dates at once
cutoff_dates = []
for year in years:
    for month, day in quarter_cutoffs:
        cutoff_dates.append({
            'cutoff_date': pd.Timestamp(year=year, month=month, day=day),
            'quarter': f"{year}{month:02d}"
        })

cutoff_df = pd.DataFrame(cutoff_dates)

In [8]:
# Filter out rows with NaN downside_deviation and prepare data
data_clean = data[data['downside_deviation'].notna()][['date', 'co_name', 'downside_deviation']].copy()

# Get unique companies
companies = data_clean['co_name'].unique()

In [9]:
# Create cross join of all cutoffs and companies
cutoff_company = cutoff_df.merge(
    pd.DataFrame({'co_name': companies}), how='cross'
)

In [10]:
cutoff_company = cutoff_company.sort_values(['cutoff_date', 'co_name']).reset_index(drop=True)
data_clean = data_clean.sort_values(['date', 'co_name']).reset_index(drop=True)

downside_df = pd.merge_asof(
    cutoff_company,
    data_clean,
    left_on='cutoff_date',
    right_on='date',
    by='co_name',
    direction='backward',
    tolerance=pd.Timedelta(days=10)  # Only match data within 10 days of cutoff
)

In [11]:
downside_df = downside_df[downside_df['downside_deviation'].notna()][['quarter', 'co_name', 'downside_deviation']]
downside_df = downside_df.sort_values(['quarter', 'co_name']).reset_index(drop=True)

In [12]:
downside_df.to_csv('../data/downside_deviation.csv', index=False)